# Lesson 2: one-bit teleportation, measurement bases, and feedforward

This is the single most important calculation in MBQC; everything else is this, repeated.

Put an arbitrary state $|\psi\rangle = a|0\rangle + b|1\rangle$ on qubit 1 and $|+\rangle$ on qubit 2, apply CZ:
$$CZ\,|\psi\rangle|+\rangle = a|0\rangle|+\rangle + b|1\rangle|-\rangle.$$
Now measure qubit 1 in a basis in the $XY$ plane of the Bloch sphere,
$$|\pm_\theta\rangle = \tfrac{1}{\sqrt2}\left(|0\rangle \pm e^{i\theta}|1\rangle\right),\qquad \text{outcome } s=0 \text{ for } +,\ s=1 \text{ for } -.$$
Projecting onto $\langle \pm_\theta|$ leaves qubit 2 in $a|+\rangle \pm e^{-i\theta} b|-\rangle$, i.e.
$$\boxed{\;|\text{out}\rangle = X^{s}\, H\, P(-\theta)\, |\psi\rangle = X^s J(\theta)|\psi\rangle\;}\qquad P(\phi)=\mathrm{diag}(1,e^{i\phi}).$$

Three lessons hide in the box:
1. **The state moved** from qubit 1 to qubit 2 (that's why it's called teleportation).
2. **A gate was applied**, $J(\theta)=HP(-\theta)$, and *we chose it* by choosing the measurement angle.
3. **There's a random Pauli** $X^s$, the *byproduct*. We know $s$, so we can undo it: that's **feedforward**.

(See [Raussendorf, Browne & Briegel, arXiv:quant-ph/0301052](https://arxiv.org/abs/quant-ph/0301052) and the tutorial
[Browne & Briegel, arXiv:quant-ph/0603226](https://arxiv.org/abs/quant-ph/0603226).)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mbqc import *

rng = np.random.default_rng(2026)

In [2]:
psi = random_state(1, rng)
theta = 0.9

for s in (0, 1):
    state = QubitState.product({1: psi, 2: PLUS})
    state.cz(1, 2)
    state.measure_xy(1, theta, outcome=s)
    expected = np.linalg.matrix_power(X, s) @ J(theta) @ psi
    print(f"outcome {s}: fidelity with X^s J(theta)|psi> = {fidelity(state.vector(), expected):.12f}")

outcome 0: fidelity with X^s J(theta)|psi> = 1.000000000000
outcome 1: fidelity with X^s J(theta)|psi> = 1.000000000000


## Why the outcome is random, and why that's good

Each outcome has probability exactly 1/2 for *every* input state. That's essential: if the outcome depended on
$|\psi\rangle$, the measurement would have learned something about $|\psi\rangle$ and disturbed it.

In [3]:
for _ in range(3):
    psi = random_state(1, rng)
    state = QubitState.product({1: psi, 2: PLUS})
    state.cz(1, 2)
    branches = [np.tensordot(v.conj(), state.amps, axes=([0], [0])) for v in xy_basis(theta)]
    print("P(s=0), P(s=1) =", [round(float(np.vdot(b, b).real), 12) for b in branches])

P(s=0), P(s=1) = [0.5, 0.5]
P(s=0), P(s=1) = [0.5, 0.5]
P(s=0), P(s=1) = [0.5, 0.5]


## Without feedforward you get noise

If we ignore $s$, the output is $J|\psi\rangle$ half the time and $XJ|\psi\rangle$ the other half: a mixed state.
In Bloch-vector terms, the $Y$ and $Z$ components are wiped out. Feedforward is not optional.

In [4]:
def bloch(v):
    return np.real([np.vdot(v, P @ v) for P in (X, Y, Z)])

psi = random_state(1, rng)
ideal = J(theta) @ psi
uncorrected = 0.5 * (bloch(ideal) + bloch(X @ ideal))
print("ideal Bloch vector       ", np.round(bloch(ideal), 4))
print("ignoring the outcome     ", np.round(uncorrected, 4))

ideal Bloch vector        [-0.848  -0.3564  0.3921]
ignoring the outcome      [-0.848  0.     0.   ]


## The same thing in PennyLane: mid-circuit measurement + feedforward

This is where PennyLane genuinely helps: `pennylane.ftqc.measure_arbitrary_basis` measures in an $XY$-plane basis
mid-circuit, and `qml.cond` applies a gate conditioned on the outcome. Its angle convention matches ours. We can't
return the state vector of a circuit with mid-circuit measurements, so we compare Bloch vectors.

In [5]:
import pennylane as qml
from pennylane.ftqc import measure_arbitrary_basis, diagonalize_mcms

dev = qml.device("default.qubit")


def teleport(correct):
    @diagonalize_mcms
    @qml.qnode(dev, mcm_method="tree-traversal")
    def circuit():
        qml.StatePrep(psi, wires=0)
        qml.H(1)
        qml.CZ([0, 1])
        m = measure_arbitrary_basis(0, angle=theta, plane="XY")
        if correct:
            qml.cond(m, qml.X)(1)  # feedforward
        return [qml.expval(op(1)) for op in (qml.X, qml.Y, qml.Z)]

    return np.array(circuit())


print("PennyLane, with feedforward   ", np.round(teleport(True), 4))
print("PennyLane, without feedforward", np.round(teleport(False), 4))
print("NumPy ideal                   ", np.round(bloch(ideal), 4))

PennyLane, with feedforward    [-0.848  -0.3564  0.3921]
PennyLane, without feedforward [-0.848  0.     0.   ]
NumPy ideal                    [-0.848  -0.3564  0.3921]


## Two steps: the byproduct changes the next angle

Chain three qubits $1-2-3$ and measure 1 then 2. After the first measurement qubit 2 holds $X^{s_1}J(\theta_1)|\psi\rangle$.
The second step sees an extra $X^{s_1}$ in front, and $X$ doesn't commute with $P(-\theta_2)$:
$$P(-\theta)\,X = X\,P(+\theta)\quad(\text{up to phase}).$$
So if $s_1=1$ we must **measure qubit 2 at $-\theta_2$ instead of $\theta_2$**. This is "adaptive measurement".
It is the only reason MBQC has a time order at all.

The rule fits in four lines (`PauliFrame` in `mbqc/patterns.py`). Keep the byproduct as $X^xZ^z$; after measuring at
angle $(-1)^x\theta$ with outcome $s$:
$$J(\theta)\,X^xZ^z = Z^xX^z\,J(\theta) \;\Rightarrow\; (x, z)\ \to\ (z\oplus s,\ x).$$

In [6]:
theta1, theta2 = 0.4, 1.3
target = J(theta2) @ J(theta1) @ psi

for adaptive in (True, False):
    fids = [fidelity(run_wire(psi, [theta1, theta2], outcomes=o, adaptive=adaptive)[0], target)
            for o in [(0, 0), (0, 1), (1, 0), (1, 1)]]
    print(f"adaptive={adaptive!s:5}: fidelity in the four outcome branches", np.round(fids, 4))

adaptive=True : fidelity in the four outcome branches [1. 1. 1. 1.]
adaptive=False: fidelity in the four outcome branches [1.     1.     0.0994 0.0994]


**Takeaways**
* Measuring in the $XY$ plane at angle $\theta$ teleports the state one site along and applies $J(\theta)=HP(-\theta)$.
* Each outcome comes with a known Pauli byproduct; we track it instead of fixing it immediately.
* Byproducts flip the sign of later angles. That dependence is feedforward, and it fixes the time order.